# CityOps Copilot — Agent Memory Workshop

A field assistant for the inspectors who keep a city's infrastructure safe — bridges, substations, pipelines, water-treatment plants, seawalls, and more. When an inspector is on site and writes up what they see, the copilot interprets the narrative, suggests a likely diagnosis, and points to the relevant history for that asset — so the inspector reasons from the full record, not just today's visit.

What makes it more than a chatbot is **memory**. It retains every asset's inspection history — across inspectors and across visits. So when a new inspector encounters a corrosion concern on Harbor Bridge, the copilot already knows what the prior inspector observed, recommended, and graded — without anyone telling it.

Built on Oracle AI Database + the `oracleagentmemory` SDK, seeded with ~220 inspection findings across 26 urban infrastructure assets (bridges, substations, pipelines, water treatment, sensors, comms towers, seawalls), plus realistic maintenance narratives that the SDK distills into memory.

>  Open `docs/overview.md` for the one-pager.

## The Data & Memory Layer

The copilot stands on **two kinds of state**, and telling them apart is the heart of the design:

- **Domain data** the agent *reads* — the facts of the physical world, stored in two plain SQL tables we load ourselves.
- **Memory** the agent *forms* — what inspectors said and what was learned from it, stored and grown by the SDK.

### 1 · Domain data — the system of record

The agent reads these tables; it never invents them. Both are ordinary Oracle tables, bulk-loaded from the committed datasets.

**`CITY_ASSET`** — the asset registry. One row per piece of infrastructure.

| asset_id | asset_class |
|---|---|
| `Harbor Bridge` | `bridge` |

**`CITY_INSPECTION_FINDING`** — structured findings, one row per observation, with a `VECTOR(384)` column for semantic search.

| finding_id | asset_id | inspector | grade | category | severity | description | recommendation | embedding |
|---|---|---|---|---|---|---|---|---|
| `a1b2c3d4e5f6` | `Harbor Bridge` | `Evelyn H. Mercer` | `B` | `corrosion` | `medium` | *"Surface corrosion and pitting on steel bearing assemblies at Pier 2; ~25% section loss…"* | *"Remove loose corrosion products, apply inhibiting primer within 60 days…"* | `[0.021, -0.044, …]` ← 384 dims |

### 2 · Memory — what the agent accumulates (SDK)

This is state the agent *forms* from interaction, not data we load. Unlike the domain tables, the SDK creates and owns both memory tables — `CITY_MESSAGE` and `CITY_MEMORY` — for you; you never write their DDL. It all comes from one call — `thread.add_messages(...)` — which produces **two distinct things**:

- **Conversational history** → stored verbatim in `CITY_MESSAGE`, scoped to that asset's thread. The raw record of *what each inspector said*, in order. **Thread-local.**
- **Extracted records** → the SDK's extractor LLM reads each narrative and distills typed, reusable memories into `CITY_MEMORY`:
  - **`fact`** — durable realities (*"Pier 2 bearings show recurring section loss"*)
  - **`preference`** — stable choices or styles an inspector defaults to
  - **`guideline`** — reusable rules and "next time do X" lessons
  
  These are **reusable across threads and inspectors** — which is exactly why a *new* inspector benefits from a *prior* inspector's experience.

## Design

Every copilot turn is a **read-then-write cycle** over a single Oracle connection: pull context from the data and memory layers, ask the LLM, then persist — and that write is what grows memory for next time.

### The big picture

The copilot reads from four stores and writes back to two, all on one database. The dashed arrow is the SDK's extractor turning a raw narrative into reusable memory.

```mermaid
flowchart TB
    INS(["Inspector"]) -->|"narrative"| COP["CityOps Copilot<br/>(call_copilot)"]
    COP <-->|"prompt + answer"| LLM[/"Agent LLM (Ollama)"/]
    COP -->|"answer"| INS

    subgraph ORA["Oracle AI Database — single connection"]
      subgraph DOM["Domain data · system of record (READ)"]
        ASSET[("CITY_ASSET")]
        FIND[("CITY_INSPECTION_FINDING<br/>VECTOR(384)")]
      end
      subgraph MEM["SDK memory (READ + WRITE)"]
        MSG[("CITY_MESSAGE")]
        MEMT[("CITY_MEMORY<br/>fact / preference / guideline")]
      end
    end

    COP -->|"asset lookup"| ASSET
    COP -->|"vector search"| FIND
    COP -->|"context card"| MSG
    COP -->|"context card"| MEMT
    COP ==>|"add_messages (WRITE)"| MSG
    MSG -.->|"extractor distills"| MEMT
```

### A copilot turn, end to end

The reads assemble the prompt; the LLM answers; the write closes the loop by feeding the next turn.

```mermaid
sequenceDiagram
    actor I as Inspector
    participant C as call_copilot
    participant DB as Oracle AI Database
    participant L as Agent LLM

    I->>C: narrative (asset + inspector)
    C->>DB: resolve thread (SDK)
    C->>DB: get_asset() · CITY_ASSET
    C->>DB: get_context_card() · CITY_MESSAGE + CITY_MEMORY
    C->>DB: find_similar_findings() · VECTOR_DISTANCE on CITY_INSPECTION_FINDING
    C->>L: system prompt + assembled context
    L-->>C: diagnosis
    C->>DB: add_messages() · write CITY_MESSAGE
    Note over DB: extractor distills new<br/>fact / preference / guideline → CITY_MEMORY
    C-->>I: answer
```

#### Worked example

Say an inspector arrives at **Harbor Bridge** — an asset a *different* inspector visited weeks earlier — and types:

> *"Rust bleed near a pier on the south side, and what looks like spalling on the concrete pedestal below."*

Here's what each step pulls in, with real values:

| Diagram step | What flows in this turn |
|---|---|
| `get_asset()` | `Harbor Bridge` → class `bridge` |
| `get_context_card()` | the asset's earlier thread messages — e.g. a prior note to *coordinate remediation with the Q3 deck resurfacing* |
| `find_similar_findings()` | top hit = an earlier logged finding — *Pier 2 bearing corrosion, ~25% section loss, grade `C`, "apply inhibiting primer within 60 days"* |
| Agent LLM | ties the new rust bleed → the documented Pier 2 corrosion; cites the prior grade and recommendation |
| `add_messages()` (write) | this narrative + the answer go to `CITY_MESSAGE`; the extractor may distill a fresh `fact` for next time |

The new inspector's first-ever look at this bridge is already shaped by an earlier inspection — **no human handoff**. That's the read path consuming exactly what an earlier write path produced.

### How it ties together

When the copilot answers, it combines both halves: it **looks up** the asset and its similar prior findings from the SQL tables (the world's current state), and it **recalls** the thread's conversation plus the extracted facts and guidelines from the SDK (accumulated experience). Domain data grounds it in what *is*; memory grounds it in what's been *learned*.

Because the **write path feeds the read path**, every inspection makes the next one sharper — even when it's a different inspector at the same asset. That feedback loop is what the rest of this workshop builds, one piece at a time.

### How we'll build it

You won't build the copilot all at once. Parts 3–5 each **build one component and verify it in isolation**; Part 6 **assembles** them into `call_copilot`. Nothing here is throwaway — every piece is wired into the final turn.

| Part | Component you build | Proven by | Powers in `call_copilot` |
|---|---|---|---|
| **3** | Conversational memory + auto-extraction (`report_event`) | TODO 2–3 checkpoints | the context card + the write-back step |
| **4** | Structured findings + vector search (`log_finding`, `find_similar_findings`) | TODO 4–5 checkpoints | the "similar past findings" retrieval |
| **5** | Scoping (user / agent / thread) | scope assertions | safe reads across many inspectors |
| **6** | **Assemble** all three into `call_copilot` | end-to-end smoke test | — |

# Part 1: Oracle Setup

Oracle AI Database is reachable at the configured DSN. We use the same
connect pattern as the `rag_to_agents` lab: hardcoded `DB_USER` / `DB_PASSWORD` /
`DB_DSN` constants set in the next cell, then a single `oracledb.connect(...)`
call — no `.env`, no wallet logic.

The SDK creates its tables under prefix `CITY_` in Part 2.

In [ ]:
# Display helpers (shared style with the rag_to_agents lab).
# ok() prints a green check so you can tell at a glance that a cell finished,
# even after the In [*] marker scrolls off-screen.
import json
import html
import decimal
from typing import Any, Iterable
from IPython.display import HTML, display


def ok(msg: str) -> None:
    """Render a green check + message as cell output."""
    display(HTML(
        f"<span style='color:#1a7f37;font-weight:700'>&#10003;</span>"
        f" <span style='color:#1a7f37'>{html.escape(msg)}</span>"
    ))


def _json_default(obj: Any) -> Any:
    if isinstance(obj, decimal.Decimal):
        return float(obj)
    if hasattr(obj, "isoformat"):
        return obj.isoformat()
    raise TypeError(f"Not JSON-serializable: {type(obj).__name__}")


def show_table(headers: list, rows: Iterable, max_width: int = 80) -> None:
    """Render a rowset as a simple HTML table."""
    def cell(v: Any) -> str:
        s = "" if v is None else str(v)
        if len(s) > max_width:
            s = s[: max_width - 1] + "\u2026"
        return html.escape(s)
    thead = "".join(
        f"<th style='text-align:left;padding:4px 10px;border-bottom:1px solid #ccc'>{html.escape(h)}</th>"
        for h in headers
    )
    body = []
    for r in rows:
        tds = "".join(
            f"<td style='padding:4px 10px;border-bottom:1px solid #eee;vertical-align:top'>{cell(v)}</td>"
            for v in r
        )
        body.append(f"<tr>{tds}</tr>")
    display(HTML(f"<table style='border-collapse:collapse'>{thead}{''.join(body)}</table>"))


def print_json(result: Any) -> None:
    """Pretty-print a JSON-ish result, handling Oracle Decimal and dates."""
    if isinstance(result, (bytes, bytearray)):
        result = result.decode("utf-8")
    if isinstance(result, str):
        try:
            result = json.loads(result)
        except json.JSONDecodeError:
            print(result)
            return
    print(json.dumps(result, indent=2, default=_json_default))


ok("Helpers loaded: ok, show_table, print_json")

In [ ]:
# === CONFIGURATION - UPDATE THESE VALUES AS NEEDED ===

# Oracle AI Database 26ai
DB_USER     = "prism"
DB_PASSWORD = "CHANGE_ME"
DB_DSN      = "aidbfree:1521/FREEPDB1"

# In-database ONNX embedding model (preloaded into Oracle via UTL).
# Same model used by the rag_to_agents lab — produces 384-dim vectors.
ONNX_MODEL  = "ALL_MINILM_L12_V2"

# Ollama — local LLM for both auto-extraction and the copilot.
OLLAMA_BASE_URL = "http://ollama:11434"
OLLAMA_MODEL    = "llama3.2"

ok(f"Config ready: db={DB_USER}@{DB_DSN}  |  onnx={ONNX_MODEL}  |  llm={OLLAMA_MODEL}")

In [ ]:
import oracledb

# Oracle 26ai's native JSON datatype round-trips as Python dict/list via
# python-oracledb. fetch_lobs=False keeps CLOB columns coming back as str
# instead of LOB locators so the rest of the notebook stays simple.
oracledb.defaults.fetch_lobs = False


def connect_to_oracle():
    """Connect to Oracle AI Database using the DB_* config from Part 0."""
    return oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)

In [ ]:
vector_conn = connect_to_oracle()
ok(f"Connected — using user {vector_conn.username}")

 Connected. Next: wire up the SDK's embedder + create the schema.

>  **Key insight — Part 1:** Oracle AI Database is a converged engine. The SDK's tables, our hand-rolled `CITY_ASSET` / `CITY_INSPECTION_FINDING` tables, and any other SQL all live on this same connection. Vector search via `VECTOR_DISTANCE()` works across all of them.

###  Optional: Reset The Workspace

If you've run this workshop before, the SDK's tables (`CITY_THREAD`, `CITY_MESSAGE`, `CITY_MEMORY`, `CITY_RECORD_CHUNKS`, `CITY_ACTOR_PROFILE`) and the workshop's custom tables (`CITY_ASSET`, `CITY_INSPECTION_FINDING`) may still hold state from prior runs.


**Run the cell below to wipe everything CITY_*, then re-run from Part 2.** If you've already initialised the SDK (`memory = OracleAgentMemory(...)`) in Part 2, you'll need to **restart the kernel** after this reset — the in-memory `memory` and `thread` objects still reference the dropped tables.

Skip this cell on the first-ever run.

In [ ]:
# Optional: wipe all CITY_* tables for a clean slate.
# Safe to run multiple times; safe to skip.
_to_drop = (
    # Drop the hand-rolled tables first (they have FKs into CITY_ASSET).
    "CITY_INSPECTION_FINDING", "CITY_ASSET",
    # Then the SDK's tables (FK-cascaded in this order).
    "CITY_MEMORY", "CITY_MESSAGE", "CITY_RECORD_CHUNKS",
    "CITY_THREAD", "CITY_ACTOR_PROFILE",
    "CITY_ORACLEAGENTMEMORY_SCHEMA_META",
)
with vector_conn.cursor() as cur:
    for t in _to_drop:
        try:
            cur.execute(f'DROP TABLE "{t}" CASCADE CONSTRAINTS')
            print(f"  dropped {t}")
        except Exception:
            pass  # table didn't exist — fine
vector_conn.commit()
ok("Workspace clean. RESTART THE KERNEL before re-running Part 2 if you've already done it.")

# Part 2: The Embedder and SDK

In [ ]:
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from langchain_oracledb.embeddings.oracleai import OracleEmbeddings
import numpy as np
import asyncio


class OracleONNXEmbedder(IEmbedder):
    """Bridges Oracle's in-database ONNX embeddings to the SDK's IEmbedder.

    Embeddings are computed inside the database via VECTOR_EMBEDDING() using
    the loaded ONNX model — same provider used by the rag_to_agents lab.
    Zero text egress, no API key, no per-call cost.
    """

    def __init__(self, conn, model_name: str = ONNX_MODEL):
        self.provider = OracleEmbeddings(
            conn=conn, params={"provider": "database", "model": model_name}
        )

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        vectors = self.provider.embed_documents(list(texts))
        return np.asarray(vectors, dtype=np.float32)

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


In [ ]:
# PASS: Checkpoint: TODO 1
embedder = OracleONNXEmbedder(vector_conn)
_v = embedder.embed(["corrosion on bearing assembly"])
assert _v.shape == (1, 384), f"Expected (1, 384), got {_v.shape}"
assert _v.dtype == np.float32, f"Expected float32, got {_v.dtype}"
ok("TODO 1 passed — embedder returns float32 (n, 384) arrays")

### SDK Initialization (Pre-Built)

Wires up `OracleAgentMemory` with a **local Ollama** model as the SDK's
internal LLM (no API key, runs in-Codespace). No changes needed.

- `extract_memories=True` turns on automatic LLM extraction of `fact` / `preference` / `guideline` / `memory` on every `add_messages` call.
- `table_name_prefix="CITY_"` namespaces the SDK's tables.

In [ ]:
from oracleagentmemory.core import OracleAgentMemory, SchemaPolicy
from oracleagentmemory.core.llms.llm import Llm
from oracleagentmemory.apis.thread import Message

# The SDK's Llm adapter speaks LiteLLM. Ollama exposes an OpenAI-compatible
# API at /v1, so we route through LiteLLM's openai/ provider pointed at the
# local Ollama server. The api_key is a required placeholder that Ollama ignores.
sdk_llm = Llm(
    model=f"openai/{OLLAMA_MODEL}",
    api_base=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama",
)

memory = OracleAgentMemory(
    connection=vector_conn,
    embedder=embedder,
    llm=sdk_llm,
    extract_memories=True,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="CITY_",
)
ok("OracleAgentMemory ready. Tables under prefix CITY_*")

# Part 3: City Asset + Auto-Extraction

> **Component 1 of 3 · build → verify.** You'll build `report_event` and watch the SDK auto-extract typed memories from raw narratives. You verify it in isolation here; the copilot wires it in at Part 6 — as the context card and the write-back step.

Two pieces in this part:

1. The **`CITY_ASSET` table** — 26 real urban infrastructure assets, loaded from `data/maintenance_logs.json` + `data/inspection_reports.json`. Hand-rolled SQL table outside the SDK.
2. **Auto-extraction from real maintenance narratives** — the SDK's extractor turns paragraph-long inspection write-ups into typed `fact` / `preference` / `guideline` records inside `CITY_MEMORY`.

Both pre-built cells run automatically; the TODOs are the auto-extraction logic.

In [ ]:
# Pre-built: load the real asset registry from the committed JSON files.
# The 26 assets are the union of asset_name fields across both datasets.
import json
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
with open(_data_dir / "inspection_reports.json") as f:
    _reports = json.load(f)

_asset_names = sorted(set(x["asset_name"] for x in _logs) | set(x["asset_name"] for x in _reports))
print(f"Loaded {len(_logs)} maintenance logs, {len(_reports)} inspection reports.")
print(f"Discovered {len(_asset_names)} unique assets.")

In [ ]:
# Pre-built: classify each asset by name heuristics into one of 8 asset classes.
def classify_asset(name: str) -> str:
    n = name.lower()
    if "bridge" in n or "overpass" in n: return "bridge"
    if "substation" in n:                  return "substation"
    if "pipeline" in n:                    return "pipeline"
    if "water" in n or "outfall" in n or "treatment" in n: return "water"
    if "solar" in n or "gas distribution" in n:           return "energy"
    if "sensor" in n or "array" in n or "monitor" in n or "gauge" in n: return "sensor"
    if "tower" in n or "relay" in n:      return "comms"
    if "seawall" in n or "retaining" in n or "booster" in n: return "civil"
    return "other"

equipment = [{"asset_id": name, "asset_class": classify_asset(name)} for name in _asset_names]
from collections import Counter
print("Asset class distribution:", dict(Counter(e['asset_class'] for e in equipment)))
print("\nFirst 5 assets:")
for e in equipment[:5]:
    print(f"  {e['asset_id']:40} -> {e['asset_class']}")

In [ ]:
# Pre-built: create PLANT_ASSET... wait, no — CITY_ASSET — and bulk-INSERT all 26.
with vector_conn.cursor() as cur:
    try:
        cur.execute("DROP TABLE CITY_INSPECTION_FINDING CASCADE CONSTRAINTS")
    except Exception:
        pass
    try:
        cur.execute("DROP TABLE CITY_ASSET CASCADE CONSTRAINTS")
    except Exception:
        pass
    cur.execute("""
        CREATE TABLE CITY_ASSET (
            asset_id     VARCHAR2(128) PRIMARY KEY,
            asset_class  VARCHAR2(32) NOT NULL,
            created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.executemany(
        "INSERT INTO CITY_ASSET (asset_id, asset_class) VALUES (:1, :2)",
        [(e["asset_id"], e["asset_class"]) for e in equipment],
    )
vector_conn.commit()
ok(f"Inserted {len(equipment)} rows into CITY_ASSET.")

In [ ]:
# Helper: look up one asset row as a dict.
def get_asset(asset_id: str) -> dict | None:
    with vector_conn.cursor() as cur:
        cur.execute(
            "SELECT asset_id, asset_class FROM CITY_ASSET WHERE asset_id = :id",
            id=asset_id,
        )
        row = cur.fetchone()
    if not row:
        return None
    return {"asset_id": row[0], "asset_class": row[1]}

print(get_asset("Harbor Bridge"))

### Auto-Extraction

When you call `thread.add_messages(...)` with `extract_memories=True`, the SDK runs its extractor LLM internally. It reads the new message + cached running summary + past memories, and outputs typed records (`fact` / `preference` / `guideline` / `memory`) into `CITY_MEMORY`.

```mermaid
flowchart LR
    MSG["New message<br/>add_messages(...)"]
    MSG --> RAW[("CITY_MESSAGE<br/>verbatim")]
    MSG --> EX{{"Extractor LLM<br/>(the SDK's own model)"}}
    SUM["Running thread summary"] -.->|advisory| EX
    PAST[("Past memories")] -.->|dedup / update| EX
    EX --> REC["2–8 atomic typed records<br/>fact · preference · guideline · memory"]
    REC --> MEM[("CITY_MEMORY")]
```

> **Heads-up — the running thread summary is active in this lab.** We create threads with `enable_context_summary=True` and `context_summary_update_frequency=2`, so the SDK folds recent turns into a compact (≤2-sentence) summary every 2 messages, using the same Ollama model. It lives on the thread row (`CITY_THREAD.runtime_config`) — *not* in `CITY_MESSAGE` — and surfaces as the `<summary>` block of the context card you'll see in Part 6.

**You don't write the extraction prompt.** The SDK does.

**In this TODO you learn:** how to write into the SDK so the extractor runs automatically. The get-or-create thread boilerplate is pre-built; you write the two lines that actually trigger extraction.

**TODO 2: write the `content` string and the `add_messages_async` call — 2 lines.** The content must include both asset and inspector (the extractor only sees text, not metadata): `f"[Asset: {asset_id}] [Inspector: {inspector}] {narrative}"`. Then `return await t.add_messages_async([Message(role="user", content=content)])`.

> **Why `_async`?** Jupyter runs cells inside an asyncio event loop. The SDK's sync methods (`add_messages`, `get_context_card`, etc.) detect this and emit a deadlock warning. The `_async` variants are the same APIs without that risk — use them with `await` everywhere in this notebook.

In [ ]:
async def report_event(asset_id: str, inspector: str, narrative: str, thread_id: str) -> list:
    """Persist a maintenance event narrative and trigger SDK auto-extraction."""
    try:
        t = memory.get_thread(thread_id)
    except Exception:
        t = None
    if t is None:
        t = memory.create_thread(
            user_id=inspector,
            thread_id=thread_id,
            agent_id="CITY",
            # Keep a rolling <=2-sentence thread summary, refreshed every 2 messages.
            enable_context_summary=True,
            context_summary_update_frequency=2,
        )
    content = f"[Asset: {asset_id}] [Inspector: {inspector}] {narrative}"
    return await t.add_messages_async([Message(role="user", content=content)])


In [ ]:
# PASS: Checkpoint: TODO 2 — smoke test with one real narrative from the dataset
_seed = _logs[0]  # First maintenance log — likely Harbor Bridge routine inspection
ids = await report_event(
    asset_id=_seed["asset_name"],
    inspector="smoke_inspector",
    narrative=_seed["narrative"],
    thread_id="smoke_thread",
)
assert ids and len(ids) >= 1, "TODO 2 incomplete — add_messages should return at least one ID"
ok(f"TODO 2 passed — added message(s): {ids}")

### See What Got Persisted

When `report_event` ran, two things happened inside the SDK:

1. **One row INSERTed into `CITY_MESSAGE`** — the raw narrative
2. **The extractor LLM ran** and INSERTed 0+ typed records into `CITY_MEMORY`

The helper below shows both tables. Run it now to see the smoke-test state, then again after TODO 3.

In [ ]:
def peek_sdk_tables(thread_id: str = None) -> None:
    """Print rows from CITY_MESSAGE and CITY_MEMORY (optionally scoped to one thread)."""
    where = "WHERE thread_id = :tid" if thread_id else ""
    bind = {"tid": thread_id} if thread_id else {}
    with vector_conn.cursor() as cur:
        cur.execute(f"""
            SELECT message_role, SUBSTR(content, 1, 100) AS preview, user_id, agent_id
              FROM CITY_MESSAGE {where}
             ORDER BY order_seq
        """, bind)
        msgs = cur.fetchall()
        print(f" CITY_MESSAGE — {len(msgs)} row(s)" + (f" for thread={thread_id}" if thread_id else ""))
        for role, preview, uid, aid in msgs:
            text = preview.read() if hasattr(preview, 'read') else preview
            print(f"   [{role:9}] user={uid or '-':18} agent={aid or '-':6} | {text}")

        cur.execute(f"""
            SELECT memory_type, SUBSTR(content, 1, 100) AS preview, user_id, agent_id
              FROM CITY_MEMORY {where}
             ORDER BY order_seq
        """, bind)
        mems = cur.fetchall()
        print(f"\n CITY_MEMORY — {len(mems)} row(s)" + (f" for thread={thread_id}" if thread_id else ""))
        for mtype, preview, uid, aid in mems:
            text = preview.read() if hasattr(preview, 'read') else preview
            print(f"   [{mtype:10}] user={uid or '-':18} agent={aid or '-':6} | {text}")

peek_sdk_tables(thread_id="smoke_thread")

**In this TODO you learn:** how scope-matching works on `memory.search` — a feature, not a quirk. Records inherit `user_id` from the thread you wrote them with, so a search must use that same `user_id` to see them; that's the multi-tenancy guarantee doing its job. The loop is pre-built; you fill in the `user_id` argument on the search call.

**TODO 3: write the value of `user_id` on the `memory.search` call — 1 line.** Pass the same inspector string the loop above wrote with (`"inspector_demo"`). If you leave it as the default (`user_id=None`) the SDK interprets that as "match records where user_id IS NULL" and you'll see zero results.

In [ ]:
# Pre-built helper: stratified sample of 8 narratives.
def sample_narratives(n: int = 8) -> list:
    """Stratified sample: mix of severities, multiple assets."""
    import random
    random.seed(42)  # Reproducible across re-runs
    buckets = {"routine": [], "warning": [], "critical": []}
    for log in _logs:
        buckets.get(log["severity"], []).append(log)
    # Take roughly proportional sample
    n_routine, n_warning, n_critical = max(1, n // 2), max(1, n // 3), max(1, n - n // 2 - n // 3)
    picks = (random.sample(buckets["routine"], min(n_routine, len(buckets["routine"]))) +
             random.sample(buckets["warning"], min(n_warning, len(buckets["warning"]))) +
             random.sample(buckets["critical"], min(n_critical, len(buckets["critical"]))))
    return picks[:n]

narratives = sample_narratives(8)
print("Sampled narratives (asset, severity):")
for n in narratives:
    print(f"  {n['asset_name']:40} [{n['severity']}]")

In [ ]:
for narr in narratives:
    await report_event(
        asset_id=narr["asset_name"],
        inspector="inspector_demo",
        narrative=narr["narrative"],
        thread_id="inspect_demo",
    )

# The SDK's high-level search requires a specific user_id (rejects
# exact_user_match=False). The records inherit user_id from the thread,
# which we created with inspector="inspector_demo".
results = await memory.search_async(
    query="recurring asset concerns and inspector practices",
    user_id="inspector_demo",
    agent_id="CITY",
    record_types=["fact", "preference", "guideline", "memory"],
    max_results=30,
)
for r in results:
    print(f"  [{r.record.record_type:11s}] {r.record.content}")


In [ ]:
# PASS: Checkpoint: TODO 3
# The SDK's high-level search REQUIRES a specific user_id (it rejects
# exact_user_match=False). We wrote the records with inspector="inspector_demo",
# so we search for the same.
_results = await memory.search_async(
    query="recurring asset concerns and inspector practices",
    user_id="inspector_demo",
    agent_id="CITY",
    record_types=["fact", "preference", "guideline", "memory"],
    max_results=50,
)
assert len(_results) >= 5, (
    f"TODO 3 — expected at least 5 extracted records from 8 narratives, got {len(_results)}.\n"
    "Check that Ollama is reachable and that you called report_event for all 8 narratives."
)
ok(f"TODO 3 passed — {len(_results)} memories extracted from 8 narratives")

### Peek At The Tables Again — After 8 Narratives

Same helper, this time scoped to the `inspect_demo` thread. You should see 8 user-role messages in `CITY_MESSAGE` and dozens of typed records in `CITY_MEMORY`. Notice that none of the records are custom types — the SDK rejects them. All extracted records are one of the four natives.

In [ ]:
peek_sdk_tables(thread_id="inspect_demo")

# Part 4: Inspection Findings + Similar-Finding Search

> **Component 2 of 3 · build → verify.** You'll build `log_finding` and `find_similar_findings`, then test the vector search on its own. The copilot calls it at Part 6 to surface similar past findings on the same asset.

An **inspection finding** isn't a `fact` or a `preference` — it's a structured domain object with `category`, `severity`, `description`, `recommendation`, `inspector`, and `overall_grade`. The SDK rejects custom `record_type` values, so findings live in their own hand-rolled `CITY_INSPECTION_FINDING` SQL table with a `VECTOR(384)` column + HNSW index.

Oracle's converged DB lets us mix vector similarity with relational filters (asset, category, severity, time window) in **one SQL statement** — no metadata-filter quirks, full SQL expressiveness.

>  Open `docs/part-4-findings-and-search.md` for the full walk-through.

In [ ]:
# Pre-built: create CITY_INSPECTION_FINDING with VECTOR(384) + HNSW index.
with vector_conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE CITY_INSPECTION_FINDING (
            finding_id      VARCHAR2(64) PRIMARY KEY,
            asset_id        VARCHAR2(128) NOT NULL,
            inspector       VARCHAR2(128),
            overall_grade   VARCHAR2(2),
            category        VARCHAR2(32),
            severity        VARCHAR2(16),
            description     CLOB NOT NULL,
            recommendation  CLOB,
            days_ago        NUMBER,
            embedding       VECTOR(384) NOT NULL,
            created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            CONSTRAINT fk_finding_asset FOREIGN KEY (asset_id) REFERENCES CITY_ASSET(asset_id)
        )
    """)
    try:
        cur.execute("""
            CREATE VECTOR INDEX city_finding_embedding_idx
            ON CITY_INSPECTION_FINDING (embedding)
            ORGANIZATION INMEMORY NEIGHBOR GRAPH
            DISTANCE COSINE
            WITH TARGET ACCURACY 95
            PARAMETERS (TYPE HNSW, M 16, EFCONSTRUCTION 200)
        """)
    except Exception as e:
        print(f"  (skipped HNSW index: {e})")
vector_conn.commit()
ok("CITY_INSPECTION_FINDING created with VECTOR(384) embedding column.")

In [ ]:
# Pre-built: bulk-load all ~220 findings from the inspection reports.
# No LLM extraction needed — findings are already structured. Just embed + INSERT.
import array, uuid

rows = []
for report in _reports:
    asset_id = report["asset_name"]
    inspector = report["inspector"]
    grade = report["overall_grade"]
    days_ago = report["days_ago"]
    for finding in report["findings"]:
        vec = array.array('f', embedder.embed([finding["description"]])[0].tolist())
        rows.append({
            "finding_id":     str(uuid.uuid4())[:12],
            "asset_id":       asset_id,
            "inspector":      inspector,
            "overall_grade":  grade,
            "category":       finding["category"],
            "severity":       finding["severity"],
            "description":    finding["description"],
            "recommendation": finding["recommendation"],
            "days_ago":       days_ago,
            "embedding":      vec,
        })

with vector_conn.cursor() as cur:
    cur.executemany("""
        INSERT INTO CITY_INSPECTION_FINDING
          (finding_id, asset_id, inspector, overall_grade, category, severity,
           description, recommendation, days_ago, embedding)
        VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                :description, :recommendation, :days_ago, :embedding)
    """, rows)
vector_conn.commit()
ok(f"Inserted {len(rows)} findings into CITY_INSPECTION_FINDING.")

**In this TODO you learn:** **the VECTOR-binding pitfall — the one real pitfall in this lab.** `oracledb` interprets a Python list as "batch values for one bind" instead of "one vector value" — passing `[0.1, 0.2, ...]` triggers `ORA-01484`. The fix is `array.array('f', ...)` (the `'f'` typecode marks float32). Everything else (UUID, INSERT, commit) is pre-built.

**TODO 4: write the line that computes `vec` — 1 line.** Embed the `description` and wrap the result in `array.array('f', ...)`:

```python
vec = array.array('f', embedder.embed([description])[0].tolist())
```

In [ ]:
import array, uuid

def log_finding(
    asset_id: str,
    inspector: str,
    category: str,
    severity: str,
    description: str,
    recommendation: str = "",
    overall_grade: str = None,
    days_ago: int = 0,
) -> str:
    """Persist a new inspection finding into CITY_INSPECTION_FINDING."""
    finding_id = str(uuid.uuid4())[:12]
    # array.array('f', ...) is what oracledb needs for VECTOR binds.
    # A plain Python list would trigger ORA-01484.
    vec = array.array('f', embedder.embed([description])[0].tolist())
    with vector_conn.cursor() as cur:
        cur.execute(
            """INSERT INTO CITY_INSPECTION_FINDING
                 (finding_id, asset_id, inspector, overall_grade, category, severity,
                  description, recommendation, days_ago, embedding)
               VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                       :description, :recommendation, :days_ago, :embedding)""",
            finding_id=finding_id, asset_id=asset_id, inspector=inspector,
            overall_grade=overall_grade, category=category, severity=severity,
            description=description, recommendation=recommendation,
            days_ago=days_ago, embedding=vec,
        )
    vector_conn.commit()
    return finding_id


In [ ]:
# PASS: Checkpoint: TODO 4
_fid = log_finding(
    asset_id="Harbor Bridge",
    inspector="checkpoint_test",
    category="corrosion",
    severity="medium",
    description="Surface corrosion on pier 2 bearing assemblies; ~25% section loss observed.",
    recommendation="Remove corrosion, apply primer + finish coat within 60 days.",
    overall_grade="C",
    days_ago=0,
)
assert _fid and isinstance(_fid, str), "TODO 4 — should return a finding_id string"

with vector_conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM CITY_INSPECTION_FINDING WHERE inspector = :i",
        i="checkpoint_test",
    )
    n = cur.fetchone()[0]
assert n == 1, "TODO 4 — checkpoint test finding not retrievable"
ok(f"TODO 4 passed — finding_id={_fid}")

**In this TODO you learn:** the converged-DB SQL pattern that mixes vector similarity with relational filters in one statement. The embedding wrap, cursor execution, and CLOB materialisation are pre-built; you write the SQL itself.

**TODO 5: write the SQL string — ~7 lines.** The pattern:

```sql
SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
       description, recommendation, days_ago,
       VECTOR_DISTANCE(embedding, :q, COSINE) AS score
  FROM CITY_INSPECTION_FINDING
 WHERE (:asset_id IS NULL OR asset_id = :asset_id)
   AND (:category IS NULL OR category = :category)
 ORDER BY score
 FETCH FIRST :k ROWS ONLY
```

The two `(:bind IS NULL OR col = :bind)` patterns are the converged-DB trick: one query handles both "search all assets" (when `asset_id` is `None`) and "narrow to one asset" (when it's set). Same for `category`.

In [ ]:
def find_similar_findings(description: str, asset_id: str = None, category: str = None, k: int = 3) -> list:
    """Vector-search CITY_INSPECTION_FINDING, optionally narrowed to one asset and/or category.

    Returns a list of dicts with all the structured fields + a `score` (cosine distance).
    """
    import array
    query_vec = array.array('f', embedder.embed([description])[0].tolist())
    sql = """
        SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
               description, recommendation, days_ago,
               VECTOR_DISTANCE(embedding, :q, COSINE) AS score
          FROM CITY_INSPECTION_FINDING
         WHERE (:asset_id IS NULL OR asset_id = :asset_id)
           AND (:category IS NULL OR category = :category)
         ORDER BY score
         FETCH FIRST :k ROWS ONLY
    """
    with vector_conn.cursor() as cur:
        cur.execute(sql, q=query_vec, asset_id=asset_id, category=category, k=k)
        cols = [d[0].lower() for d in cur.description]
        rows = []
        for r in cur.fetchall():
            row = dict(zip(cols, r))
            for key in ("description", "recommendation"):
                v = row.get(key)
                if v is not None and hasattr(v, "read"):
                    row[key] = v.read()
            rows.append(row)
    return rows


In [ ]:
# PASS: Checkpoint: TODO 5
_broad = find_similar_findings("bearing corrosion at piers", k=5)
_bridge = find_similar_findings("bearing corrosion at piers", asset_id="Harbor Bridge", k=5)
_corrosion_only = find_similar_findings("bearing corrosion at piers", category="corrosion", k=5)

assert _broad and len(_broad) >= 3, f"TODO 5 — broad search returned {len(_broad) if _broad else 0} hits"
assert _bridge and all(r["asset_id"] == "Harbor Bridge" for r in _bridge), \
    "TODO 5 — asset_id filter should restrict results to Harbor Bridge"
assert _corrosion_only and all(r["category"] == "corrosion" for r in _corrosion_only), \
    "TODO 5 — category filter should restrict results to corrosion only"
ok(f"TODO 5 passed — broad={len(_broad)}, asset-filtered={len(_bridge)}, category-filtered={len(_corrosion_only)}")
for r in _bridge[:3]:
    print(f"  score={r['score']:.3f}  [{r['category']}/{r['severity']}]  {str(r['description'])[:90]}")

# Part 5: Scoping — Inspector vs City

> **Component 3 of 3 · build → verify.** You'll prove that one inspector can't read another's private notes — the multi-tenant safety guarantee the copilot depends on every turn.

Three scope dimensions: `user_id` (inspector), `agent_id` (`CITY`), `thread_id` (per asset). Enforced as SQL `WHERE` predicates — cross-user leakage is impossible at the DB layer.

**No TODO** — just run the demo cells to see scoping in action.

In [ ]:
# Inspector Mercer writes a personal note (user-scoped, invisible to others)
memory.add_memory(
    content="Remember to swap shifts with Jordan next Tuesday.",
    user_id="Evelyn_H_Mercer",
)

# Mercer also writes a city-wide tribal-knowledge guideline (agent-scoped)
memory.add_memory(
    content="On Harbor Bridge, inspect Pier 2 bearings annually — corrosion-prone since 2024.",
    agent_id="CITY",
)
ok("Mercer wrote one personal memory and one city-wide memory.")

In [ ]:
# Inspector Vance searches at user scope — should NOT see Mercer's personal note
vance_personal = await memory.search_async(
    query="shift swap notes",
    user_id="Jordan_Vance",
    record_types=["memory"],
    max_results=10,
)

# Vance searches at city scope — SHOULD see the Pier 2 guideline
vance_city = await memory.search_async(
    query="Harbor Bridge Pier 2 bearings",
    user_id=None,           # explicitly leave user dimension unconstrained
    agent_id="CITY",
    record_types=["memory"],
    max_results=10,
)

print(f"  Vance's personal-scope hits for 'shift swap': {len(vance_personal)}  (should be 0)")
print(f"  Vance's city-scope hits for 'Pier 2 bearings': {len(vance_city)}  (should be ≥ 1)")

In [ ]:
# Assertions — multi-tenancy is enforced at the SQL layer
assert all("shift swap" not in r.record.content.lower() for r in vance_personal), \
    "Cross-inspector leak: Mercer's personal note showed up in Vance's user-scoped search"
assert any("Pier 2" in r.record.content for r in vance_city), \
    "Pier 2 guideline not retrievable at agent scope — check agent_id wiring"
ok("Multi-tenancy verified: Vance sees city-wide guidelines but NOT Mercer's personal notes.")

>  **Key insight — Part 5:** scoping is enforced as a SQL `WHERE` clause on `user_id` / `agent_id` / `thread_id` columns — not as a soft filter in Python. A bug in your harness can't leak Mercer's note to Vance; only a SQL injection could. For regulated infrastructure, add VPD policies on top (see `docs/part-5-scoping.md`).

# Part 6: The CityOps Copilot — End-to-End

> **Assembly.** Every component you built and verified in Parts 3–5 now snaps together into a single `call_copilot` turn — context card, similar-finding search, scoped reads, and the write-back that grows memory.

One function, `call_copilot`, ties together: thread resolution (SDK), asset lookup (`CITY_ASSET` SQL), the context card (SDK), similar-finding search (`CITY_INSPECTION_FINDING` SQL via `VECTOR_DISTANCE()`), agent LLM call, and persistence (which triggers auto-extraction).

>  Open `docs/part-6-copilot-end-to-end.md` for the architecture diagram.

In [ ]:
# Pre-built: system prompt + LLM call helper.
from types import SimpleNamespace
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

COPILOT_SYSTEM_PROMPT = """You are a CityOps inspection copilot.

Each turn you are given:
- The current inspection narrative
- The asset record (class)
- A thread context card (recent inspector messages + extracted facts/guidelines)
- Up to 3 similar past findings on the same asset (with category, severity,
  recommendation, prior inspector, prior overall grade)

Your job: suggest a likely diagnosis or characterisation; cite prior findings
with their inspector + grade + recommendation timeline; surface relevant
guidelines from the thread context. Keep responses ≤ 8 sentences.
When safety- or maintenance-critical guidelines apply, name them."""

# Same ChatOllama setup as the rag_to_agents lab. temperature=0 keeps the
# cross-inspector handoff demo deterministic across re-runs.
_chat_llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)


async def call_ollama_chat(messages: list, model: str = None):
    """Call the local Ollama model with a simple chat messages list.

    Async so it composes with the SDK's async methods inside Jupyter's
    event loop. Returns a minimal chat-completion-shaped object
    so callers can read `.choices[0].message.content` unchanged.
    """
    role_to_cls = {
        "system": SystemMessage,
        "user": HumanMessage,
        "assistant": AIMessage,
    }
    lc_msgs = [role_to_cls[m["role"]](content=m["content"]) for m in messages]
    resp = await _chat_llm.ainvoke(lc_msgs)
    return SimpleNamespace(choices=[
        SimpleNamespace(message=SimpleNamespace(content=resp.content))
    ])

### What Does `get_context_card` Actually Return?

Before you wire it into `call_copilot` (TODO 6), see what the SDK gives you. The cell below grabs the `inspect_demo` thread (8 narratives from Part 3) and prints `card.formatted_content` verbatim.

The card is **XML, not Markdown**. Four blocks: `<summary>`, `<topics>`, `<relevant_information>`, `<recent_messages>`.

In [ ]:
_demo_thread = memory.get_thread("inspect_demo")
_demo_card = await _demo_thread.get_context_card_async(
    fallback_message_count=100,
    max_recent_messages=5,
    max_relevant_results=5,
)
print("=" * 70)
print("Raw context-card output — this is the XML the agent LLM will see:")
print("=" * 70)
print(_demo_card.formatted_content)

**In this TODO you learn:** the two **SDK-facing reads** that power the cross-inspector handoff. Thread resolution, asset lookup, context assembly, the LLM call, and persistence are all pre-built — they're mechanical glue. You write only the two lines that pull in prior context.

**TODO 6: write 2 lines.**

1. `card = await t.get_context_card_async(fallback_message_count=100, max_recent_messages=10, max_relevant_results=8)` — SDK conversational state
2. `similar = find_similar_findings(narrative, asset_id=asset_id, k=3)` — SQL-backed prior findings (sync — our own helper, not an SDK call)

Together these are what make Vance's copilot turn aware of Mercer's work with zero human handoff.

In [ ]:
async def call_copilot(narrative: str, inspector_id: str, thread_id: str, asset_id: str) -> str:
    """End-to-end CityOps copilot turn: build context, query LLM, persist."""
    # 1. Resolve thread.
    try:
        t = memory.get_thread(thread_id)
    except Exception:
        t = None
    if t is None:
        t = memory.create_thread(
            user_id=inspector_id,
            thread_id=thread_id,
            agent_id="CITY",
            enable_context_summary=True,
            context_summary_update_frequency=2,
        )

    # 2. Asset record — straight SQL lookup against CITY_ASSET.
    asset = get_asset(asset_id)
    asset_info = (
        f"Asset {asset['asset_id']} (class: {asset['asset_class']})"
        if asset else "(no asset record found)"
    )

    # 3. Context card (thread state + extracted facts/guidelines).
    card = await t.get_context_card_async(
        fallback_message_count=100,
        max_recent_messages=10,
        max_relevant_results=8,
    )

    # 4. Similar past findings on this asset.
    similar = find_similar_findings(narrative, asset_id=asset_id, k=3)
    if similar:
        similar_text = "\n\n".join(
            f"  (score={r['score']:.3f})  [{r['category']}/{r['severity']}]  "
            f"inspector={r['inspector']}, grade={r['overall_grade']}, days_ago={r['days_ago']}\n"
            f"     description: {r['description']}\n"
            f"     recommendation: {r['recommendation']}"
            for r in similar
        )
    else:
        similar_text = "  (no prior findings for this asset)"

    # 5. Assemble the context string.
    context = (
        f"# Current inspection narrative\n"
        f"Asset: {asset_id}\n"
        f"Inspector: {inspector_id}\n"
        f"Narrative: {narrative}\n\n"
        f"# Asset record\n{asset_info}\n\n"
        f"# Thread context\n{card.formatted_content}\n\n"
        f"# Similar past findings (from CITY_INSPECTION_FINDING)\n{similar_text}"
    )

    # 6. LLM call.
    # Print the user message so it's clear exactly what we send to the model.
    print("=" * 70)
    print("USER MESSAGE sent to the model:")
    print("=" * 70)
    print(context)
    print("=" * 70 + "\n")

    messages = [
        {"role": "system", "content": COPILOT_SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    resp = await call_ollama_chat(messages)
    answer = resp.choices[0].message.content or ""

    # 7. Persist — both messages trigger SDK auto-extraction.
    await t.add_messages_async([
        Message(role="user",      content=f"[{inspector_id} @ {asset_id}] {narrative}"),
        Message(role="assistant", content=answer),
    ])
    return answer


In [ ]:
# PASS: Checkpoint: TODO 6 — smoke test
_smoke = await call_copilot(
    narrative="Smoke test — please ignore. One-line check on Harbor Bridge.",
    inspector_id="smoke_inspector",
    thread_id="copilot_smoke",
    asset_id="Harbor Bridge",
)
assert _smoke and len(_smoke) > 10, "TODO 6 — copilot returned empty/short answer"
ok("TODO 6 passed — copilot ran end-to-end")
print(f"\nSample response (first 300 chars):\n  {_smoke[:300]}")

## The Cross-Inspector Handoff Scenario

Inspector Mercer reviews Harbor Bridge, logs a corrosion finding. Days later, Inspector Vance — who has never met Mercer — encounters a related concern on the same asset. Watch what the copilot tells Vance.

### Day 1 — Mercer's Visit Uses Both Functions, In Sequence

The two functions you built in Parts 4 and 6 play **complementary roles** in a real inspection workflow. Mercer's day-1 visit shows both:

| Step | Function | Role | Cost |
|---|---|---|---|
| 1 | `call_copilot(...)` | Reasoning interface. Mercer types what she sees; the copilot interprets, suggests, surfaces any prior findings on this asset. | ~4 LLM calls, 30–60 s |
| 2 | `log_finding(...)` | System of record. Mercer decides what the finding is and stores it as a structured row with category, severity, recommendation, grade. | 0 LLM calls, ~100 ms |

**Why both?** `call_copilot` is the diagnostic dialogue; `log_finding` is the formal record. The structured row Mercer logs in step 2 becomes **searchable evidence** for the next inspector's `call_copilot` invocation — via `find_similar_findings` inside the copilot — closing the loop.

Below: each function gets its own cell.

In [ ]:
# Step 1 — Mercer reports what she sees and asks the copilot for help.
# This is the REASONING call: builds a context card, vector-searches
# CITY_INSPECTION_FINDING for prior issues on Harbor Bridge, and runs the
# agent LLM. On this first visit there are no prior findings, so the
# copilot's response is mostly fresh diagnosis.
print("=" * 70)
print("DAY 1, step 1: Mercer calls call_copilot for diagnostic help")
print("=" * 70)
MERCER_NOTES = await call_copilot(
    narrative=(
        "Quarterly inspection of Harbor Bridge. Surface corrosion observed on Pier 2 "
        "bearing assemblies (south side), estimated section loss ~25% on bearing plate "
        "edges with local rust bleeding onto the concrete pedestal. Corrosion extends "
        "roughly 1.5 m longitudinally along the bearing line."
    ),
    inspector_id="Evelyn_H_Mercer",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\nMercer's copilot response:\n{MERCER_NOTES}")

Now Mercer has decided this is a real corrosion concern. The copilot's answer helped her think; the structured fields are her conclusion.

**Step 2** is the recording call — `log_finding` writes one row into `CITY_INSPECTION_FINDING` with the embedding computed locally on the `description`.

This row is what `find_similar_findings` will surface for the next inspector.

In [ ]:
# Step 2 — Mercer formally records the finding she's now confident about.
# Pure persistence: one INSERT + one local embedding. Fast and free.
MERCER_FINDING_ID = log_finding(
    asset_id="Harbor Bridge",
    inspector="Evelyn_H_Mercer",
    category="corrosion",
    severity="medium",
    description=(
        "Surface corrosion + pitting on steel bearing assemblies at Pier 2 south, "
        "~25% section loss; rust bleed onto concrete pedestal; ~1.5m longitudinal extent."
    ),
    recommendation=(
        "Remove loose corrosion products, apply corrosion-inhibiting primer + finish "
        "coat to affected bearing assemblies within 60 days; re-inspect annually with "
        "caliper section-loss measurements."
    ),
    overall_grade="C",
    days_ago=0,
)
print(f"Logged finding {MERCER_FINDING_ID} — now searchable by future call_copilot invocations.")

In [ ]:
# Day 1, 14:00 — Mercer adds a follow-up observation
print("\n" + "=" * 70)
print("DAY 1 (afternoon): Mercer follow-up note")
print("=" * 70)
MERCER_FOLLOWUP = await call_copilot(
    narrative=(
        "Coordinated with maintenance — recommend scheduling the bearing remediation "
        "to coincide with the deck wearing-surface resurfacing in Q3 to share access "
        "and traffic management."
    ),
    inspector_id="Evelyn_H_Mercer",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\n Mercer's followup response:\n{MERCER_FOLLOWUP}")

### Day N — Vance Only Calls `call_copilot`. Why?

Vance is in the diagnostic phase — he's seen rust bleed and possible spalling, but he hasn't decided what the finding is yet. He calls `call_copilot` to **interpret** what he's seeing, not to record anything.

Under the hood, `call_copilot` does two things that pull in Mercer's prior work without anyone telling it to:

1. **`find_similar_findings(...)`** vector-searches `CITY_INSPECTION_FINDING` for Harbor Bridge — Mercer's logged finding (the row from the cell above) is the top hit. The copilot's prompt gets her category, severity, recommendation, and grade as structured fields.
2. **`thread.get_context_card(...)`** assembles Mercer's earlier messages + any auto-extracted facts/guidelines/preferences from those messages. So Vance gets her conversational reasoning + soft observations (like *coordinate with Q3 deck resurfacing*) that the formal finding doesn't carry.

If Vance later confirms his diagnosis, he'd add a step-2 `log_finding(...)` of his own — but the workshop stops here because the magic moment is Vance's reasoning being shaped by Mercer's prior work without human handoff.

In [ ]:
# Day N (later) — Inspector Vance arrives. Different inspector, same asset.
# Only call_copilot — Vance is still in diagnosis mode, not yet recording.
print("\n" + "=" * 70)
print("DAY N: Inspector Vance on Harbor Bridge (never met Mercer)")
print("=" * 70)
VANCE_DIAGNOSIS = await call_copilot(
    narrative=(
        "Reviewing Harbor Bridge as part of routine cycle. Noticing rust bleed near "
        "a pier on the south side and what looks like spalling on the concrete "
        "pedestal below."
    ),
    inspector_id="Jordan_Vance",
    thread_id="asset_harbor_bridge",
    asset_id="Harbor Bridge",
)
print(f"\nVance's copilot response (built from Mercer's work, NO human handoff):\n{VANCE_DIAGNOSIS}")

## Compare: Vance With And Without Memory

Below: same Vance narrative, but stripped of all memory layers — no thread, no context card, no `find_similar_findings`. Just the LLM and the narrative. The contrast is the workshop's point.

In [ ]:
stateless_messages = [
    {"role": "system", "content": COPILOT_SYSTEM_PROMPT},
    {"role": "user",   "content": (
        "Reviewing Harbor Bridge as part of routine cycle. Noticing rust bleed near "
        "a pier on the south side and what looks like spalling on the concrete "
        "pedestal below."
    )},
]
STATELESS_VANCE = (await call_ollama_chat(stateless_messages)).choices[0].message.content

print("=" * 70)
print("WITHOUT MEMORY (stateless LLM):")
print("=" * 70)
print(STATELESS_VANCE)
print("\n" + "=" * 70)
print("WITH MEMORY (the copilot you built):")
print("=" * 70)
print(VANCE_DIAGNOSIS)

## Key Takeaways

| Layer | Earned its keep by… |
|---|---|
| Auto-extracted `fact` / `preference` / `guideline` (SDK) | Surfacing tribal knowledge from real maintenance narratives without explicit code |
| `CITY_ASSET` SQL table | Letting the copilot look up structured asset facts at the start of every turn |
| `CITY_INSPECTION_FINDING` SQL with `VECTOR(384)` | Vector-searchable history via Oracle's native `VECTOR_DISTANCE()` — mixed with relational filters in one SQL |
| Context card (SDK) | Compressing the per-asset thread into ~200 tokens that travel with every turn |
| Scoping (`user_id` / `agent_id` / `thread_id`) | Multi-tenant-safety at the SQL layer |

## Where to Next?

- **[Oracle AI Agent Memory documentation](https://docs.oracle.com/en/database/oracle/agent-memory/)**
- **[Oracle AI Developer Hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)**
- **[Agent Memory short course (DeepLearning.AI)](https://www.deeplearning.ai/short-courses/agent-memory-building-memory-aware-agents/)**